# YOLO 合并模型测试

本notebook用于测试合并后的YOLO模型的性能和属性

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

import warnings

warnings.filterwarnings("ignore")

print(plt.rcParams["font.sans-serif"])

# 设置中文字体，matplotlib 对 ttc 的支持有点问题，只能识别到 Noto Sans CJK JP 字体，但是这个字体包含了中文字符，所以可以正常显示中文。
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP"]
plt.rcParams["axes.unicode_minus"] = False  # 解决负号显示问题

# 设置路径
PROJECT_ROOT = Path(f"{os.environ['HOME']}/tp_yolo")
sys.path.insert(0, str(PROJECT_ROOT))


print("✓ 环境配置完成")

In [ ]:
import matplotlib.font_manager as fm

# 打印所有包含 "Noto" 的字体信息
for f in fm.fontManager.ttflist:
    if 'Noto' in f.name:
        print(f.name, '->', f.fname)
    elif 'SC' in f.name:
        print(f.name, '->', f.fname)

## 1. 加载模型和数据集配置

In [ ]:
# 查找最佳模型
model_base_dir = PROJECT_ROOT / "runs" / "detect" / "runs"
model_dir = model_base_dir / "merged" / "merged_yolo_det"
best_model_path = model_dir / "weights" / "best.pt"
last_model_path = model_dir / "weights" / "last.pt"

# 检查模型是否存在
if best_model_path.exists():
    print(f"✓ 发现最佳模型: {best_model_path}")
    model = YOLO(str(best_model_path))
    model_used = "best"
elif last_model_path.exists():
    print(f"⚠ 未找到best.pt，使用 last.pt: {last_model_path}")
    model = YOLO(str(last_model_path))
    model_used = "last"
else:
    # 尝试加载预训练模型（如果尚未训练）
    print("⚠ 未找到本地训练的模型，使用预训练的 yolo11m.pt")
    model = YOLO("yolo11m.pt")
    model_used = "pretrained"

# use_device = '1,2,3' if model_used != "pretrained" else '0'
use_device = "0"

print(f"✓ 模型加载完成 (使用: {model_used})")

# 加载数据集配置
data_yaml = PROJECT_ROOT / "data" / "merged_yolo_detect" / "data.yaml"
if data_yaml.exists():
    import yaml

    with open(data_yaml) as f:
        data_config = yaml.safe_load(f)
    print("✓ 数据集配置加载完成")
    print(f"  - 类别数: {data_config['nc']}")
    print(f"  - 类别名: {data_config['names']}")
    classes = data_config["names"]
    nc = data_config["nc"]
else:
    print("⚠ 未找到data.yaml")
    classes = None
    nc = 8

## 2. 模型架构和性能统计

In [ ]:
# 模型信息
print(f"{'='*60}")
print("模型信息")
print(f"{'='*60}")
print(f"模型名称: {model.model_name}")
print("模型类型: Detection")
print(f"设备: {model.device}")
print(f"参数数量: {sum(p.numel() for p in model.model.parameters()) / 1e6:.1f}M parameters")

# 模型复杂度
model.info(verbose=False)

# 输入输出形状
dummy_input_shape = (1, 3, 640, 640)
print(f"\n输入形状: {dummy_input_shape} (batch_size, channels, height, width)")
print("输出: 模型输出检测结果 (boxes, confidences, class_ids)")

# 通过现有数据测试合并后训练的模型

In [ ]:
import time

all_time = []


def test_inference(model, image_path):
    print(f"\n正在测试推理性能，使用图像: {image_path}")
    start = time.perf_counter()
    result = model(image_path, device=use_device)
    end = time.perf_counter()
    all_time.append(end - start)
    print(f"\n推理时间: {end - start:.3f} 秒")

    print("\n检测结果:")
    if len(result[0].boxes) == 0:
        print("  - 未检测到任何对象")
    else:
        print(f"  - 检测框数量: {len(result[0].boxes)}")
        print(f"  - 检测框坐标: {result[0].boxes.xyxy[0].cpu().numpy()}")
        print(f"  - 置信度: {result[0].boxes.conf[0].cpu().numpy()}")
        print(f"  - 类别ID: {result[0].boxes.cls[0].cpu().numpy()}")

    # 可视化检测结果
    test_image_path = Path(image_path)

    if test_image_path.exists():
        img = cv2.imread(str(test_image_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        # 原图
        axes[0].imshow(img_rgb)
        axes[0].set_title("原始图像", fontsize=12, fontweight="bold")
        axes[0].axis("off")

        # 检测结果图
        result_img = result[0].plot()
        result_img_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
        axes[1].imshow(result_img_rgb)
        axes[1].set_title("检测结果", fontsize=12, fontweight="bold")
        axes[1].axis("off")

        plt.show()


current_dir = Path.cwd()
test_images_dir = current_dir / "test_images"

all_images = list(test_images_dir.glob("*.png"))
print(f"✓ 发现 {len(all_images)} 张测试图像:")
all_images.sort(key=lambda x: x.name)  # 按文件名排序

for img_file in all_images:
    test_inference(model, img_file)

print(f"\n平均推理时间: {sum(all_time) / len(all_time):.3f} 秒")
print(f"总推理时间: {sum(all_time):.3f} 秒")

# 测试级联专家模型

In [ ]:
from MoEYOLO.moe_yolo.cascade import CascadeMoEPipeline
from MoEYOLO.moe_yolo.config import CascadeConfig, ExpertConfig, TriggerConfig


def init_cascade(
    class_names: list, base_path: Path = model_base_dir / "moeyolo"
) -> CascadeMoEPipeline:
    base_ckpt = base_path / "moe_v1_base" / "weights" / "best.pt"
    ground_expert_ckpt = base_path / "moe_v1_ground_expert" / "weights" / "best.pt"
    tiny_expert_ckpt = base_path / "moe_v1_tiny_obstacle_expert" / "weights" / "best.pt"
    config = CascadeConfig(
        base_model_path=str(base_ckpt),
        class_names=tuple(class_names),
        trigger=TriggerConfig(
            low_conf_threshold=0.35,
            low_conf_ratio_trigger=0.50,
            tiny_area_ratio=0.01,
            tiny_count_trigger=4,
            max_experts_per_frame=2,
        ),
    )

    experts = [
        ExpertConfig(
            name="ground_expert",
            model_path=str(ground_expert_ckpt),
            focus_classes=(0, 1),
            conf=0.20,
            iou=0.60,
        ),
        ExpertConfig(
            name="tiny_obstacle_expert",
            model_path=str(tiny_expert_ckpt),
            focus_classes=(2, 3, 4, 5, 6, 7),
            conf=0.20,
            iou=0.60,
        ),
    ]

    pipeline = CascadeMoEPipeline(config=config, experts=experts)
    return pipeline


class_names = (
    list(data_config["names"].values())
    if classes is not None
    else [f"class_{i}" for i in range(nc)]
)

pipeline = init_cascade(class_names)

In [ ]:
import time

all_time = []


def test_cascade_inference(pipeline: CascadeMoEPipeline, image_path):
    print(f"\n正在测试Cascade推理性能，使用图像: {image_path}")
    test_image_path = Path(image_path)
    if not test_image_path.exists():
        print(f"⚠ 图像文件不存在: {test_image_path}")
        return
    frame = cv2.imread(str(test_image_path))
    start = time.perf_counter()
    result = pipeline.infer(frame)
    end = time.perf_counter()
    all_time.append(end - start)
    print(f"\n推理时间: {end - start:.3f} 秒")

    route_decision = result.route
    detections = result.detections

    print("\n检测结果:")
    if len(detections) == 0:
        print("  - 未检测到任何对象")
    else:
        print(f"  - 路由决策: {route_decision}")
        print(f"  - 检测框数量: {len(detections)}")
        for i, det in enumerate(detections):
            print(f"  - 检测框 {i + 1}:")
            print(f"    - 坐标: {det.box}")
            print(f"    - 置信度: {det.conf:.3f}")
            print(f"    - 类别ID: {det.cls_id}")
            print(f"    - 类别名: {det.cls_name}")

    # 可视化检测结果
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    # 原图
    axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title("原始图像", fontsize=12, fontweight="bold")
    axes[0].axis("off")

    # 检测结果图
    shape = frame.shape
    for det in detections:
        norm_x1, norm_y1, norm_x2, norm_y2 = det.box
        x1 = int(norm_x1 * shape[1])
        y1 = int(norm_y1 * shape[0])
        x2 = int(norm_x2 * shape[1])
        y2 = int(norm_y2 * shape[0])

        frame = cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"{det.cls_name} {det.conf:.2f}"
        frame = cv2.putText(
            frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2
        )
    axes[1].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Cascade检测结果", fontsize=12, fontweight="bold")
    axes[1].axis("off")

    plt.show()


current_dir = Path.cwd()
test_images_dir = current_dir / "test_images"

all_images = list(test_images_dir.glob("*.png"))
print(f"✓ 发现 {len(all_images)} 张测试图像:")
all_images.sort(key=lambda x: x.name)  # 按文件名排序

for img_file in all_images:
    test_cascade_inference(pipeline, img_file)

print(f"\n平均推理时间: {sum(all_time) / len(all_time):.3f} 秒")
print(f"总推理时间: {sum(all_time):.3f} 秒")

## 3. 在测试集上验证模型

In [ ]:
# 在测试集上验证
test_image_dir = PROJECT_ROOT / 'data' / 'merged_yolo_detect' / 'images' / 'test'
if test_image_dir.exists():
    test_images = list(test_image_dir.glob('*.jpg')) + list(test_image_dir.glob('*.png'))
    print(f"测试集图像数: {len(test_images)}")
    
    # 运行验证（如果这是训练好的模型）
    if model_used != "pretrained":
        print("\n正在验证模型...")
        # 使用val模式进行完整的性能评估
        val_result = model.val(data=str(data_yaml), device=use_device)
        
        print(f"\n{'='*60}")
        print(f"验证结果")
        print(f"{'='*60}")
        if hasattr(val_result, 'results_dict'):
            for key, value in val_result.results_dict.items():
                if isinstance(value, float):
                    print(f"{key}: {value:.4f}")
        else:
            print(f"mAP50: {val_result.box.map50:.4f}")
            print(f"mAP50-95: {val_result.box.map:.4f}")
    else:
        # 检查训练结果文件
        results_csv = PROJECT_ROOT / 'runs' / 'detect' / 'merged_yolo_det' / 'results.csv'
        if results_csv.exists():
            results_df = pd.read_csv(results_csv)
            print("\n训练过程中的性能指标 (最后5个epoch):")
            print(results_df.tail())
else:
    print(f"⚠ 测试集不存在: {test_image_dir}")
    print("  可能原因：模型还未训练，或数据集路径不正确")
    
    

## 4. 推理速度测试

In [ ]:
import time

# 创建测试图像
test_image_path = PROJECT_ROOT / 'data' / 'merged_yolo_detect' / 'images' / 'test'
if test_image_path.exists():
    test_imgs = list(test_image_path.glob('*.jpg'))[:10]
else:
    test_image_path = PROJECT_ROOT / 'data' / 'merged_yolo_detect' / 'images' / 'train'
    test_imgs = list(test_image_path.glob('*.jpg'))[:10]

if test_imgs:
    print(f"正在测试推理速度 (使用{len(test_imgs)}张图像)...")
    
    # 预热
    _ = model(str(test_imgs[0]), verbose=False)
    
    # 测试推理
    inference_times = []
    for img_path in test_imgs:
        start_time = time.time()
        results = model(str(img_path), verbose=False)
        inference_time = (time.time() - start_time) * 1000
        inference_times.append(inference_time)
    
    print(f"\n{'='*60}")
    print(f"推理性能")
    print(f"{'='*60}")
    print(f"平均推理时间: {np.mean(inference_times):.2f} ms")
    print(f"最小推理时间: {np.min(inference_times):.2f} ms")
    print(f"最大推理时间: {np.max(inference_times):.2f} ms")
    print(f"标准差: {np.std(inference_times):.2f} ms")
    print(f"FPS (estimated): {1000/np.mean(inference_times):.1f}")
else:
    print("⚠ 未找到测试图像")

## 5. 样本推理和检测可视化

In [ ]:
# 运行推理并可视化结果
test_image_dir = PROJECT_ROOT / 'data' / 'merged_yolo_detect' / 'images' / 'test'
if not test_image_dir.exists():
    test_image_dir = PROJECT_ROOT / 'data' / 'merged_yolo_detect' / 'images' / 'train'

if test_image_dir.exists():
    test_images = list(test_image_dir.glob('*.jpg')) + list(test_image_dir.glob('*.png'))
    sample_images = test_images[:9]  # 采样9张图像
    
    # 创建图表
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(sample_images):
        # 运行推理
        results = model.predict(source=str(img_path), verbose=False)
        result = results[0]
        
        # 获取带标注的图像
        annotated_image = result.plot()
        
        # 显示
        axes[idx].imshow(annotated_image[..., ::-1])  # BGR to RGB
        
        # 统计检测信息
        n_detections = len(result.boxes)
        title = f"检测数: {n_detections}"
        if n_detections > 0 and classes:
            detected_classes = [classes[int(cls)] for cls in result.boxes.cls]
            class_counts = {}
            for cls_name in detected_classes:
                class_counts[cls_name] = class_counts.get(cls_name, 0) + 1
            title += f"\n{class_counts}"
        
        axes[idx].set_title(title, fontsize=10)
        axes[idx].axis('off')
    
    # 隐藏未使用的子图
    for idx in range(len(sample_images), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'test_visualization.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✓ 可视化结果已保存到 test_visualization.png")
else:
    print("⚠ 测试图像目录不存在")

## 6. 各类别性能分析

In [ ]:
# 分析各类别的检测性能
if classes:
    test_image_dir = PROJECT_ROOT / "data" / "merged_yolo_detect" / "images" / "test"
    if not test_image_dir.exists():
        test_image_dir = (
            PROJECT_ROOT / "data" / "merged_yolo_detect" / "images" / "train"
        )

    if test_image_dir.exists():
        test_images = list(test_image_dir.glob("*.jpg")) + list(
            test_image_dir.glob("*.png")
        )

        # 统计各类别检测数
        class_detection_counts = {cls: 0 for cls in classes}
        total_detections = 0
        image_with_detections = 0

        print("正在分析各类别检测性能...")
        for img_path in test_images[:100]:  # 采样100张图像进行统计
            results = model.predict(source=str(img_path), verbose=False)
            result = results[0]

            if len(result.boxes) > 0:
                image_with_detections += 1
                for cls_id in result.boxes.cls:
                    cls_id = int(cls_id)
                    if cls_id < len(classes):
                        class_name = classes[cls_id]
                        class_detection_counts[class_name] += 1
                        total_detections += 1

        print(f"\n{'=' * 60}")
        print(f"采样图像 (前100张) 的检测统计")
        print(f"{'=' * 60}")
        print(f"含有检测的图像数: {image_with_detections}")
        print(f"总检测数: {total_detections}")
        print(f"平均每图检测数: {total_detections / max(image_with_detections, 1):.2f}")

        print(f"\n各类别检测频率:")
        print(f"{'-' * 60}")

        # 创建DataFrame用于展示
        class_stats = []
        for cls_name, count in class_detection_counts.items():
            percentage = (count / total_detections * 100) if total_detections > 0 else 0
            class_stats.append(
                {"类别": cls_name, "检测数": count, "百分比": f"{percentage:.2f}%"}
            )

        stats_df = pd.DataFrame(class_stats)
        print(stats_df.to_string(index=False))

        # 绘制柱状图
        fig, ax = plt.subplots(figsize=(12, 6))
        class_names_sorted = sorted(
            class_detection_counts.keys(),
            key=lambda x: class_detection_counts[x],
            reverse=True,
        )
        counts_sorted = [class_detection_counts[cls] for cls in class_names_sorted]

        bars = ax.bar(
            class_names_sorted, counts_sorted, color="skyblue", edgecolor="navy"
        )
        ax.set_xlabel("类别", fontsize=12)
        ax.set_ylabel("检测数", fontsize=12)
        ax.set_title("各类别检测频率分析", fontsize=14, fontweight="bold")
        ax.tick_params(axis="x", rotation=45)

        # 添加数值标签
        for bar in bars:
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                height,
                f"{int(height)}",
                ha="center",
                va="bottom",
                fontsize=10,
            )

        plt.tight_layout()
        plt.savefig(
            PROJECT_ROOT / "class_detection_stats.png", dpi=100, bbox_inches="tight"
        )
        plt.show()
        print("\n✓ 统计图表已保存到 class_detection_stats.png")
else:
    print("⚠ 未能加载类别信息")

## 7. 置信度分析和检测参数分布

In [ ]:
# 收集置信度数据
test_image_dir = PROJECT_ROOT / "data" / "merged_yolo_detect" / "images" / "test"
if not test_image_dir.exists():
    test_image_dir = PROJECT_ROOT / "data" / "merged_yolo_detect" / "images" / "train"

if test_image_dir.exists():
    test_images = list(test_image_dir.glob("*.jpg")) + list(
        test_image_dir.glob("*.png")
    )

    confidences = []
    box_areas = []

    print("正在收集检测参数...")
    for img_path in test_images[:100]:
        results = model.predict(source=str(img_path), verbose=False)
        result = results[0]

        if len(result.boxes) > 0:
            # 收集置信度
            confidences.extend(result.boxes.conf.cpu().numpy().tolist())

            # 计算box面积
            boxes = result.boxes.xywh.cpu().numpy()
            for box in boxes:
                area = box[2] * box[3]  # width * height
                box_areas.append(area)

    if confidences:
        print(f"\n{'=' * 60}")
        print(f"检测置信度统计")
        print(f"{'=' * 60}")
        print(f"平均置信度: {np.mean(confidences):.4f}")
        print(f"最小置信度: {np.min(confidences):.4f}")
        print(f"最大置信度: {np.max(confidences):.4f}")
        print(f"中位数: {np.median(confidences):.4f}")
        print(f"标准差: {np.std(confidences):.4f}")

        if box_areas:
            print(f"\n检测框体积统计:")
            print(f"平均面积: {np.mean(box_areas):.2f}")
            print(f"最小面积: {np.min(box_areas):.2f}")
            print(f"最大面积: {np.max(box_areas):.2f}")

        # 绘制置信度分布
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # 直方图
        axes[0].hist(confidences, bins=30, color="skyblue", edgecolor="navy", alpha=0.7)
        axes[0].axvline(
            np.mean(confidences),
            color="red",
            linestyle="--",
            label=f"Mean: {np.mean(confidences):.4f}",
        )
        axes[0].set_xlabel("Confidence", fontsize=11)
        axes[0].set_ylabel("Frequency", fontsize=11)
        axes[0].set_title("Confidence Distribution", fontsize=12, fontweight="bold")
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        # 框体面积分布
        if box_areas:
            axes[1].hist(
                box_areas, bins=30, color="lightcoral", edgecolor="darkred", alpha=0.7
            )
            axes[1].axvline(
                np.mean(box_areas),
                color="blue",
                linestyle="--",
                label=f"Mean: {np.mean(box_areas):.2f}",
            )
            axes[1].set_xlabel("Detection Box Area (pixels²)", fontsize=11)
            axes[1].set_ylabel("Frequency", fontsize=11)
            axes[1].set_title(
                "检测框面积分布", fontsize=12, fontweight="bold"
            )
            axes[1].legend()
            axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig(
            PROJECT_ROOT / "confidence_distribution.png", dpi=100, bbox_inches="tight"
        )
        plt.show()
        print("\n✓ 置信度分析图已保存到 confidence_distribution.png")
else:
    print("⚠ 测试图像目录不存在")